<a href="https://colab.research.google.com/github/maheen-armghan/AI-ANN-and-Deep-learning-projects/blob/main/Fine_tuned_GPT_2_Art_Text_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/AI-ANN-and-Deep-learning-projects/blob/main/Fine_tuned_GPT_2_Art_Text_Generator.ipynb)

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.optim import AdamW
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments

# 1. Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')

# 2. Sample data
texts = [
    "Art is my favorite subject.",
    "Artists from history include Leonardo da Vinci, Michelangelo, Claude Monet, Mary Cassatt, Pablo Picasso and Frida Kahlo.",
    "Abstract art is weird and disturbing.",
    "Museums like the Louvre, MoMA, and the Uffizi Gallery preserve art for future generations.",
    "Street art transforms public spaces into open galleries accessible to everyone.",
    "Art therapy uses creative expression to help people process emotions and heal trauma.",
    "Indigenous art often incorporates symbols and patterns passed down through generations.",
    "Vincent van Gogh painted The Starry Night in 1889 while staying at an asylum.",
    "The Renaissance was a period of great artistic revival in Europe from the 14th to 17th century.",
    "Watercolor paintings use transparent pigments dissolved in water for soft, luminous effects.",
]

# 3. Fixed Dataset class (padding tokens masked from loss)
class SimpleDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=70):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )

    def __getitem__(self, idx):
        input_ids      = torch.tensor(self.encodings["input_ids"][idx])
        attention_mask = torch.tensor(self.encodings["attention_mask"][idx])
        labels         = input_ids.clone()
        labels[attention_mask == 0] = -100  # mask padding from loss
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels
        }

    def __len__(self):
        return len(self.encodings["input_ids"])

dataset = SimpleDataset(texts, tokenizer)

# 4. Training arguments (fixed)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=50,                # more epochs for tiny dataset
    per_device_train_batch_size=2,
    learning_rate=3e-5,                 # lower lr for stability
    warmup_steps=10,                    # gradual warmup
    weight_decay=0.01,                  # regularization
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# 5. Train
trainer.train()

# 6. Fixed generation (sampling instead of greedy)
model.eval()

input_text = "Art is my "
inputs = tokenizer(input_text, return_tensors="pt")

output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=30,          # tokens to generate AFTER the prompt
    do_sample=True,             # sampling instead of greedy decoding
    temperature=0.7,            # controls randomness
    top_k=50,                   # sample from top 50 tokens only
    top_p=0.95,                 # nucleus sampling
    repetition_penalty=1.3,     # penalize repeated tokens
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Art is my vernacular. It's what I do when things are weird and disturbing to me (and, sometimes even scary).
The first time you walk into


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!find /content -name "*.ipynb" 2>/dev/null

/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/lab4ML1260.ipynb
/content/drive/MyDrive/Colab Notebooks/lab8-1260.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
/content/drive/MyDrive/Colab Notebooks/1260ML LAB 14.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
/content/drive/MyDrive/Colab Notebooks/1260lab1ann.ipynb
/content/drive/MyDrive/Colab Notebooks/1260-NLP Text Summarization lab.ipynb
/content/drive/MyDrive/Colab Notebooks/midlab1260.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled6.ipynb
/content/drive/MyDrive/Colab Notebooks/Fine-tuned GPT-2 Art Text Generator.ipynb


In [ ]:
import nbformat

notebook_path = "/content/drive/MyDrive/Fine_tuned_GPT_2_Art_Text_Generator.ipynb"

with open(notebook_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# Remove broken widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(notebook_path, "w") as f:
    nbformat.write(nb, f)

print("Fixed! Now re-save to GitHub.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Fine_tuned_GPT_2_Art_Text_Generator.ipynb'